# Strain corrections: full tensor elasticity and Bir-Pikus coupling

This notebook supersedes the strain treatment used in `pryor_benchmark.ipynb`. **That notebook's
numbers are stale** -- it calls `qdsolver_core.trace_strain_from_mask`, which has two independent
defects, both fixed here.

Nothing below has been executed. Run top to bottom to reproduce; the measured values quoted in
the markdown come from `benchmark_corrected_strain.log` in this folder.

## The two defects

**1. Wrong by a constant factor.** `trace_strain_from_mask` returns the *constrained* (total)
strain of the Eshelby inclusion, $3\alpha\varepsilon^*=\frac{1+\nu}{1-\nu}\varepsilon^*$. The
deformation potentials $a_c$ and $a_v$ act on the *elastic* strain -- total minus eigenstrain --
which is $3(\alpha-1)\varepsilon^*=\frac{2(1-2\nu)}{1-\nu}\varepsilon^*$. The ratio

$$\frac{1+\nu}{2(1-2\nu)} = 1.1652 \quad \text{at the GaAs Voigt } \nu = 0.23501$$

so every band-edge shift computed from it was ~17% too large.

**2. No shear at all.** Only the trace exists in that model, so the Bir-Pikus $b$ and $d$ terms
could not be formed. In the Pryor $b=14$ nm pyramid the rms shear strain is ~13% of the trace,
and Pryor's Sec. VI attributes hole confinement mainly to shear.

In [ ]:
import numpy as np
import qdsolver_core as qd
import strain_fourier as sf
import kp_confined as kpc
import kp_pryor as kp
import pryor1998 as pr
import eigensolvers as eig
from qdsolver_core import HBAR2_OVER_2M0 as G

np.set_printoptions(suppress=True, precision=6)

dot, matrix = pr.PRYOR_TABLE_I['InAs'], pr.PRYOR_TABLE_I['GaAs']
eps_star = qd.eigenstrain(dot['a0'], matrix['a0'])
nu = qd.voigt_poisson_ratio(matrix['C11'], matrix['C12'], matrix['C44'])
C = (matrix['C11'], matrix['C12'], matrix['C44'])
BASE, GAAS_CB = 14.0, matrix['Eg']

print(f"eps_star = {eps_star:.6f}   Voigt nu = {nu:.5f}")
print(f"old (constrained) trace  = {qd.trace_strain_from_mask(np.array([True]), eps_star, nu)[1]:+.6f}")
print(f"correct (elastic) trace  = {sf.isotropic_sphere_trace(eps_star, nu):+.6f}")
print(f"ratio = {(1+nu)/(2*(1-2*nu)):.4f}")

## The method

`strain_fourier.solve_strain` solves linear continuum elasticity exactly in Fourier space. With a
dilatational eigenstrain $\varepsilon^*_{kl}=\varepsilon_T\delta_{kl}\chi(\mathbf r)$ confined to
the dot, mechanical equilibrium $\partial_j\sigma_{ij}=0$ becomes a 3x3 linear system per
wavevector,

$$K_{ik}(\mathbf k)\,u_k(\mathbf k) = -i\,\varepsilon_T (C_{11}+2C_{12})\,\chi(\mathbf k)\,k_i,
\qquad K_{ik}=C_{ijkl}k_jk_l$$

with $K$ the acoustic (Christoffel) tensor. Forming
$\varepsilon_{ij}=\tfrac{i}{2}(k_ju_i+k_iu_j)$ gives the complete tensor in one FFT round trip.
The shape enters only through $\chi(\mathbf k)$, so any mask from `qdsolver_core` works unchanged,
and the full anisotropic cubic $C_{11}/C_{12}/C_{44}$ are kept rather than Voigt-averaged.

This is the construction of Andreev, Downes, Faux & O'Reilly, *J. Appl. Phys.* **86**, 297 (1999).

**What is still approximate**, and stated in the module docstring: homogeneous elastic constants
(what makes the solve exact); linear elasticity at ~7% mismatch; continuum, so the result carries
the $C_{4v}$ symmetry of the pyramid rather than the true $C_{2v}$ of zincblende -- only an
atomistic valence-force-field relaxation gets that right (Pryor, Kim, Wang, Williamson & Zunger,
*J. Appl. Phys.* **83**, 2548 (1998)); and a clamped periodic cell, whose $O(f)$ error
`padding_report` measures directly.

## Validation 1: pseudomorphic / clamped slab -- exact

For a slab only $\mathbf k=(0,0,k_z)$ survives, so the solution is closed-form. Note this is a
strained **superlattice**, not a layer on a thick substrate: a clamped periodic cell shares the
mismatch with the barrier in proportion to the fill fraction $f$, and only as $f\to0$ does it
reduce to the textbook $\varepsilon_{zz}=-2(C_{12}/C_{11})\varepsilon_\parallel$.

This is the sharpest available test -- exact, and anisotropic, so it pins down the sign
convention and the elastic tensor at once. **Measured: agreement to 2.78e-17.**

In [ ]:
N, h = 32, 0.5
zc = np.arange(N) * h
print(f"{'n_layers':>8} {'f':>7} | {'exx exact':>11} {'numeric':>11} | {'ezz exact':>11} {'numeric':>11}")
for nz in (1, 2, 4, 11, 16):
    Z = np.meshgrid(zc, zc, zc, indexing='ij')[2]
    slab = Z < nz * h
    e = sf.solve_strain(slab, eps_star, *C, h)
    g = e.at(slab)
    pa, za = sf.clamped_slab_strain(eps_star, C[0], C[1], slab.mean())
    print(f"{nz:>8} {slab.mean():>7.4f} | {pa:>11.6f} {g['exx']:>11.6f} | {za:>11.6f} {g['ezz']:>11.6f}")
pl, zl = sf.pseudomorphic_layer_strain(eps_star, C[0], C[1])
print(f"{'f -> 0':>16} | {pl:>11.6f} {'':>11} | {zl:>11.6f}   <- pseudomorphic limit")

## Validation 2: isotropic Eshelby sphere

Driven into the isotropic regime the solver must reproduce Eshelby's uniform interior strain and
Davies' vanishing trace outside. The residual is purely the $O(f)$ clamping offset of the finite
cell: **measured rel. error falls 7.59e-2 -> 2.05e-3 as the box grows 24 -> 80 nm**, with
error$/f$ constant at ~1.16. Interior isotropy and near-zero core shear hold throughout.

In [ ]:
Ci = sf.isotropic_constants(nu)
tr_a = sf.isotropic_sphere_trace(eps_star, nu)
print(f"analytic elastic trace inside = {tr_a:+.6f}")
print(f"{'box(nm)':>8} {'f':>8} {'trace(core)':>13} {'rel err':>10} {'shear':>10}")
h, R = 0.5, 6.0
for N in (48, 64, 96, 128, 160):
    c = np.arange(N) * h - (N - 1) * h / 2
    X, Y, Z = np.meshgrid(c, c, c, indexing='ij')
    sph = qd.sphere_mask(X, Y, Z, R)
    es = sf.solve_strain(sph, eps_star, *Ci, h)
    core = qd.sphere_mask(X, Y, Z, R * 0.6)
    g, gr = es.at(core), es.at_rms(core)
    tr = g['exx'] + g['eyy'] + g['ezz']
    print(f"{N*h:>8.0f} {sph.mean():>8.5f} {tr:>13.6f} {abs(tr/tr_a-1):>10.2e} "
          f"{max(gr['exy'],gr['eyz'],gr['ezx']):>10.2e}")

## The Pryor pyramid: what the old model was missing

Measured at $h=0.7$, pad $=17.5$ nm:

| quantity | value |
|---|---|
| $\langle\varepsilon_{xx}\rangle=\langle\varepsilon_{yy}\rangle$ | $-0.046401$ |
| $\langle\varepsilon_{zz}\rangle$ | $+0.004746$ |
| trace | $-0.088056$ |
| biaxial $\varepsilon_{xx}+\varepsilon_{yy}-2\varepsilon_{zz}$ | $-0.102292$ |
| rms $\varepsilon_{yz}=\varepsilon_{zx}$ | $0.011528$ |
| rms $\varepsilon_{xy}$ | $0.006987$ |
| old model | trace $-0.107951$ uniform, **zero shear** |

The rms shear is ~13% of the trace. Note also that the trace $-0.0881$ is properly *more*
compressive than a pseudomorphic quantum well ($-0.0732$), as 3D confinement requires -- a useful
physical sanity check on the magnitude.

**Use `at_rms` for shear, not `at`.** Every dot shape here is symmetric under $x\to-x$, so the
mean shear vanishes identically no matter how large the local shear is, and a mean-based
diagnostic reports zero and looks like a solver bug.

In [ ]:
def grid(h, pad, base=BASE):
    cx = np.arange(-(base / 2 + pad), base / 2 + pad + 1e-9, h)
    cz = np.arange(-pad, base / 2 + pad + 1e-9, h)
    X, Y, Z = np.meshgrid(cx, cx, cz, indexing='ij')
    return X, Y, Z, qd.pyramid_mask(X, Y, Z, base)

for h, pad in ((0.7, 10.5), (0.7, 17.5), (0.5, 17.5)):
    X, Y, Z, pyr = grid(h, pad)
    ep = sf.solve_strain(pyr, eps_star, *C, h)
    d, dr = ep.at(pyr), ep.at_rms(pyr)
    print(f"h={h} pad={pad}  grid {X.shape}, f={pyr.mean():.4f}, "
          f"clamping residual {sf.padding_report(ep, pyr):.4f}")
    print(f"   trace {d['exx']+d['eyy']+d['ezz']:+.6f}  "
          f"biaxial {d['exx']+d['eyy']-2*d['ezz']:+.6f}")
    print(f"   rms shear exy={dr['exy']:.6f} eyz={dr['eyz']:.6f} ezx={dr['ezx']:.6f}   "
          f"(mean ezx = {abs(d['ezx']):.1e}, zero by symmetry)")

### Two sensitivity checks worth knowing

**Elastic-constant choice** (the main remaining approximation). Measured CB well depth:
GaAs 425.9 meV, InAs 491.4, average 452.5. The conventional matrix (GaAs) choice already gives
the shallowest well, so this knob cannot explain the residual electron overbinding below --
moving to dot constants makes agreement worse.

**Well taper.** The corrected model reproduces Pryor's Fig. 4 taper qualitatively: measured
438.7 meV at $z=1.4$ nm falling to 349.7 meV at $z=5.6$ nm, against his 400 -> 270. The old
model gave a flat 297 meV everywhere.

In [ ]:
X, Y, Z, pyr = grid(0.7, 10.5)
print(f"{'C_ij from':>14} {'trace':>9} {'biaxial':>9} {'CB well meV':>12}")
for nm, Cset in (('GaAs (matrix)', C),
                 ('InAs (dot)', (dot['C11'], dot['C12'], dot['C44'])),
                 ('average', tuple((matrix[k] + dot[k]) / 2 for k in ('C11', 'C12', 'C44')))):
    st = sf.solve_strain(pyr, eps_star, *Cset, 0.7)
    d = st.at(pyr)
    Ve, _, _, _ = pr.band_edge_fields(pyr, st.trace)
    print(f"{nm:>14} {d['exx']+d['eyy']+d['ezz']:>9.5f} "
          f"{d['exx']+d['eyy']-2*d['ezz']:>9.5f} {(GAAS_CB-Ve[pyr].mean())*1e3:>12.1f}")

st = sf.solve_strain(pyr, eps_star, *C, 0.7)
Ve, _, _, _ = pr.band_edge_fields(pyr, st.trace)
z0 = int(round(10.5 / 0.7))
print("\nwell taper, base -> apex (Pryor Fig. 4: 400 -> 270 meV):")
for zi in range(0, 11, 2):
    sl = pyr[:, :, z0 + zi]
    if sl.sum() == 0:
        continue
    print(f"   z = {zi*0.7:4.1f} nm ({sl.sum():4d} pts)  well "
          f"{(GAAS_CB - Ve[:, :, z0+zi][sl].mean())*1e3:6.1f} meV")

## Bir-Pikus coupling

The shear strain enters the 4-, 6- and 8-band Hamiltonians through `kp_pryor.bir_pikus_terms`,
which adds to the kinetic $Q$, $R$, $S$:

$$Q_\varepsilon = -\tfrac{b}{2}(\varepsilon_{xx}+\varepsilon_{yy}-2\varepsilon_{zz}), \quad
R_\varepsilon = \tfrac{\sqrt3}{2}b(\varepsilon_{xx}-\varepsilon_{yy}) - i\,d\,\varepsilon_{xy},
\quad S_\varepsilon = -d(\varepsilon_{zx}-i\,\varepsilon_{yz})$$

These are purely local (multiplicative) operators, so unlike the kinetic terms they raise no
operator-ordering question where $b$ and $d$ jump across the interface.

Reference: Bir & Pikus, *Symmetry and Strain-Induced Effects in Semiconductors* (Wiley, 1974).

### Test 1: over-determined correspondence with the kinetic terms

The strain and kinetic Hamiltonians have identical structure under
$\varepsilon_{ij}\leftrightarrow k_ik_j$. Matching term by term gives
$G\gamma_2\leftrightarrow-b/2$ and $2\sqrt3G\gamma_3\leftrightarrow-d$ -- and the *same two*
substitutions must reproduce all three of $Q$, $R$, $S$. Three complex expressions pinned by two
constants, so a misplaced or mis-phased element cannot survive.

**Measured: 6.94e-18 for 4-, 6- and 8-band.** This is the check that the earlier 6-band bug
(misplaced $R$/$S$) would have failed.

In [ ]:
kx, ky, kz = 0.21, -0.13, 0.31
st_k = sf.StrainTensor(kx*kx, ky*ky, kz*kz, kx*ky, ky*kz, kz*kx)
for nb in (4, 6, 8):
    p = dict(dot)
    if nb == 8:
        p['Ep'] = 0.0          # U and V are linear in k and have no strain analogue
        g1, g2, g3 = kp.modified_luttinger(p['gamma1L'], p['gamma2L'], p['gamma3L'],
                                           p['Eg'], p['delta_so'], p['Ep'])
    else:
        g1, g2, g3 = p['gamma1L'], p['gamma2L'], p['gamma3L']
    q = dict(p, b=-2*G*g2, d=-2*np.sqrt(3)*G*g3, a_v=G*g1, a_c=G)
    H_kin = kp.bulk_hamiltonian(kx, ky, kz, p, n_bands=nb)
    H_str = kp.bulk_hamiltonian(0, 0, 0, q, n_bands=nb, strain=st_k, include_hydrostatic=True)
    print(f"{nb}-band  max|H_BirPikus - H_kinetic| = {np.abs(H_str-H_kin).max():.2e}")

### Test 2: physical sign -- independent of Test 1

Test 1 only fixes the three terms *relative to one another*; the overall sign needs separate
physics. Under compressive biaxial strain the heavy hole must be the topmost valence band.

Measured, for InAs on GaAs ($Q_\varepsilon=-0.22929$ eV with $b=-1.8$):

- top valence state is **HH+3/2 with weight 1.0000** in all three models
- 4-band HH-LH splitting **458.6 meV, exactly $2|Q_\varepsilon|$**
- 6-band LH-SO repulsion through $Q_\varepsilon$: analytic 2x2 gives $+0.0283/-0.6376$ eV,
  numeric gives the same to 4 decimals

In [ ]:
par, zz = sf.pseudomorphic_layer_strain(eps_star, C[0], C[1])
st_b = sf.StrainTensor(par, par, zz, 0.0, 0.0, 0.0)
Qe, Re, Se = kp.bir_pikus_terms(st_b, dot['b'], dot['d'])
print(f"InAs on GaAs: exx=eyy={par:+.5f} ezz={zz:+.5f};  Q_eps={Qe.real:+.5f} eV")
for nb in (4, 6, 8):
    H = kp.bulk_hamiltonian(0, 0, 0, dot, n_bands=nb, strain=st_b)
    E, V = np.linalg.eigh(H)
    top = np.argsort(E)[-1] if nb != 8 else np.argsort(E)[-3]
    w = np.abs(V[:, top])**2
    print(f"{nb}-band  hermiticity {np.abs(H-H.conj().T).max():.1e}   "
          f"top valence state {kp.BAND_LABELS[nb][int(np.argmax(w))]} (weight {w.max():.4f})")

E4 = np.sort(np.linalg.eigvalsh(kp.bulk_hamiltonian(0, 0, 0, dot, n_bands=4, strain=st_b)))[::-1]
print(f"\n4-band HH-LH splitting {(E4[0]-E4[2])*1e3:.1f} meV = 2|Q_eps| {2*abs(Qe.real)*1e3:.1f} meV")
lh, so = Qe.real, -dot['delta_so']
disc = np.sqrt(((lh-so)/2)**2 + 2*Qe.real**2)
E6 = np.sort(np.linalg.eigvalsh(kp.bulk_hamiltonian(0, 0, 0, dot, n_bands=6, strain=st_b)))[::-1]
print(f"6-band LH-SO: analytic {(lh+so)/2+disc:+.4f} / {(lh+so)/2-disc:+.4f} eV")
print(f"              numeric  {E6[2]:+.4f} / {E6[4]:+.4f} eV")

### The double-counting trap

`confined_hamiltonian(..., include_hydrostatic=False)` is the default **on purpose**. The standard
path builds `Ev`/`Ec` with `pryor1998.band_edge_fields`, which has already applied
$a_c\mathrm{Tr}\,\varepsilon$ and $-a_v\mathrm{Tr}\,\varepsilon$. Since $P$ is built as
$-E_v+\text{kinetic}$, the hydrostatic strain is already in the Hamiltonian; setting
`include_hydrostatic=True` on top applies it twice. Set it True only when `Ev`/`Ec` are the
*unstrained* edges.

Not included, and documented as dropped: the strain renormalization of $P_0$ and the
strain-induced conduction-valence coupling of a fully general eight-band Bir-Pikus treatment.

In [ ]:
X, Y, Z, pyr = grid(2.0, 4.0)
st = sf.solve_strain(pyr, eps_star, *C, 2.0)
Ve, Vh, _, _ = pr.band_edge_fields(pyr, st.trace)
ops = kpc.GridOperators(X.shape, 2.0, periodic=False)
print(f"grid {X.shape} -- Hermiticity must survive the complex R_eps diagonal")
for nb in (4, 6, 8):
    f = kp.material_fields(pyr, dot, matrix, Vh, Ve, n_bands=nb)
    H0 = kp.confined_hamiltonian(ops, f, n_bands=nb, hole_convention=(nb != 8))
    H1 = kp.confined_hamiltonian(ops, f, n_bands=nb, hole_convention=(nb != 8), strain=st)
    print(f"  {nb}-band  hermiticity {eig.hermiticity_error(H1):.2e}   "
          f"strain contributes up to {abs(H1-H0).max():.4f} eV")

## Corrected benchmark results

Measured values below are from `benchmark_corrected_strain.log`; the cells that follow regenerate
them (slow -- the full sweep took ~40 min).

### One-band electron, $h=0.7$, pad $=14$ nm

| | well (meV) | $E_0$ (eV) | binding (meV) | states | $E_1-E_0$ |
|---|---|---|---|---|---|
| old, $m=0.023$ | 297.0 | 1.4466 | 72.4 | 1 | -- |
| **corrected, $m=0.023$** | 428.3 | 1.3768 | 142.2 | 1 | -- |
| old, $m=0.040$ | 297.0 | 1.4266 | 92.4 | 1 | -- |
| **corrected, $m=0.040$** | 428.3 | 1.3423 | 176.7 | 3 | 132.1 |
| *Pryor* | *400 -> 270* | *~1.41* | *~110* | *1* | *~110* |

Extra binding from the heavier mass: **34.5 meV corrected** vs Pryor's 30 (old model gave 20.0).
Padding-converged -- binding moves only 139.5 -> 142.2 meV between pad 10.5 and 14.0 nm.

### Multiband holes, meV above the GaAs VB edge

| model | old | **corrected + shear** | Pryor |
|---|---|---|---|
| 4-band, $h=1.4$ | 116.7 | **188.3** | ~195 |
| 4-band, $h=1.0$ | 138.3 | **189.0** | ~195 |
| 6-band, $h=1.4$ | 122.5 | **202.4** | ~235 |
| 6-band, $h=1.0$ | 148.2 | **202.6** | ~235 |
| 8-band, $h=1.4$ | 106.4 | 154.8 (soft) | ~195 |
| 8-band, $h=1.0$ | 108.0 | 148.9 (soft) | ~195 |
| 8-band electron, $h=1.0$ | 1.3818 eV | 1.2916 eV | ~1.35 eV |

The 4-band error drops from $-57$ to $-6$ meV and the 6-band from $-87$ to $-32$. Equally
important, the corrected numbers are **grid-converged** (188.3 -> 189.0, 202.4 -> 202.6) where the
old ones still drifted 20-26 meV: the shear terms are what was making hole states grid-sensitive.

In [ ]:
# One-band, corrected vs old. Slow: each solve is ~1-10 min at these sizes.
for h, pad in ((0.7, 10.5), (0.7, 14.0)):
    X, Y, Z, pyr = grid(h, pad)
    tr_new = sf.solve_strain(pyr, eps_star, *C, h).trace
    tr_old, _ = qd.trace_strain_from_mask(pyr, eps_star, nu)
    print(f"\nh={h} pad={pad}  grid {X.shape} = {X.size:,} pts")
    print(f"  trace in dot: corrected {tr_new[pyr].mean():+.5f}  old {tr_old[pyr].mean():+.5f}")
    for label, tr in (("corrected", tr_new), ("old", tr_old)):
        for case in ('unstrained', 'strain_averaged'):
            V_e, V_h, m_e, m_h = pr.band_edge_fields(pyr, tr, mass_case=case)
            E, _, _ = eig.solve_lowest(qd.build_hamiltonian(m_e, V_e, h), k=6,
                                       tol=1e-9, maxiter=6000)
            nb = int((E < GAAS_CB).sum())
            extra = f"  E1-E0={((E[1]-E[0])*1e3):7.1f} meV" if nb > 1 else ""
            print(f"  {label:>9s} m={pr.PRYOR_ELECTRON_MASSES[case]['dot']:.3f}  "
                  f"well {(GAAS_CB-V_e[pyr].mean())*1e3:6.1f} meV  E0={E[0]:.4f} eV  "
                  f"binding {(GAAS_CB-E[0])*1e3:6.1f} meV  bound={nb}{extra}")

In [ ]:
# Multiband. h=1.4 is the inline-runnable size; h=1.0 took ~25 min for the 8-band pair.
import time
for h, pad in ((1.4, 8.4),):
    X, Y, Z, pyr = grid(h, pad)
    st = sf.solve_strain(pyr, eps_star, *C, h)
    tr_old, _ = qd.trace_strain_from_mask(pyr, eps_star, nu)
    ops = kpc.GridOperators(X.shape, h, periodic=False)
    print(f"h={h} pad={pad}  grid {X.shape} = {X.size:,} pts, {8*X.size:,} unknowns (8-band)")
    for label, tr, strain in (("old (hydro only)", tr_old, None),
                              ("corrected+shear", st.trace, st)):
        V_e, V_h, _, _ = pr.band_edge_fields(pyr, tr)
        print(f"  --- {label}")
        for nb in (4, 6):
            f = kp.material_fields(pyr, dot, matrix, V_h, V_e, n_bands=nb)
            H = kp.confined_hamiltonian(ops, f, n_bands=nb, hole_convention=True, strain=strain)
            E, _, _ = eig.solve_lowest(H, k=4, tol=1e-8, maxiter=6000)
            print(f"      {nb}-band hole {(-E[0])*1e3:7.1f} meV above GaAs VB edge")
        f8 = kp.material_fields(pyr, dot, matrix, V_h, V_e, n_bands=8)
        H8 = kp.confined_hamiltonian(ops, f8, n_bands=8, strain=strain)
        Ee, _, _ = eig.solve_interior(H8, k=3, sigma=float(V_e[pyr].min()),
                                      tol=1e-7, maxiter=8000)
        Eh, _, _ = eig.solve_interior(H8, k=3, sigma=float(V_h[pyr].max()),
                                      tol=1e-7, maxiter=8000)
        print(f"      8-band electron {Ee[0]:.4f} eV   8-band hole {Eh[0]*1e3:7.1f} meV")

## What is settled, and what is not

**Settled.** The hole states. 4-band lands 6 meV from Pryor and 6-band 32 meV, both
grid-converged, where the old model was 57 and 87 meV off and still drifting. The earlier
caveat -- *"the valence band can't match Pryor with the present strain model"* -- no longer
applies.

**Open, and I would not paper over these:**

1. **Electrons moved the wrong way.** One-band $E_0$ went 1.4466 -> 1.3768 eV against Pryor's
   ~1.41; 8-band went 1.3818 -> 1.2916 eV against ~1.35. The deeper corrected well overbinds.
   The factor-1.165 correction re-derives cleanly and the internal checks pass
   ($a_g=a_v+a_c$ reproduces the gap shift exactly; the pyramid trace is properly more
   compressive than a QW), and the elastic-constant choice cannot explain it. Remaining
   suspects: the omitted piezoelectric potential, and the precision of the Fig. 4/7 values
   being compared against.

2. **4-band and 8-band disagree by 40 meV**, where Pryor states they agree within 3 meV. That is
   a signal, not noise. The 8-band solves are also the numerically weakest: the folded-spectrum
   hole solve needed 3248 iterations to reach 7.8e-07 eV, and its $h=1.4$ counterpart tripped
   the conditioning warning at 2.0e-06 eV. Treat the 8-band holes as soft. Suspects are the
   dropped $P_0$ strain renormalization and strain-induced CB-VB coupling.

3. **4-to-6-band splitting is 13.6 meV vs Pryor's ~40.** Improved from 9.9, not resolved.

**Now unblocked: piezoelectricity.** The shear components drive
$P_i=2e_{14}\varepsilon_{jk}$ ($i,j,k$ cyclic) in zincblende, whose bound charge
$-\nabla\!\cdot\!\mathbf P$ feeds straight into `poisson_solver.solve_poisson`. This was simply
not computable before -- there was no shear to feed it -- and it is the term that breaks the
p-state degeneracy of a $C_{4v}$ pyramid. Pryor's Table I carries $e_{14}$, so he includes it and
this code does not. Worth knowing before leaning on it: first-order piezoelectricity alone is now
known to be unreliable in these dots, the second-order response being comparable and often
opposing (Bester and Zunger and co-workers, c. 2006).